In [1]:
from heart_disease.data.ingestion import load_file
from heart_disease.config import (
    MODEL_DIR,
    INTERIM_DATA_DIR,
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
)
from heart_disease.models.train import load_model
from heart_disease.models.evaluation import (
    get_confusion_matrix,
    get_classification_report,
)
from heart_disease.models.interpretation import (
    get_feature_coefficients,
    get_prediction_errors,
)

In [2]:
X_test = load_file(INTERIM_DATA_DIR / "X_test.parquet")
y_test = load_file(INTERIM_DATA_DIR / "y_test.parquet")["target"]

final_lr_model = load_model(MODEL_DIR / "logistic_regression.joblib")

2026-09-19 01:45:30,378 | INFO | heart_disease.models.train | Loading LogisticRegression model from /home/irasionize/heart-disease-ml/models/logistic_regression.joblib


In [3]:
# evaluate_model(final_lr_model, X_test, y_test, ExperimentConfig())

In [4]:
get_confusion_matrix(final_lr_model, X_test, y_test)

,Predicted 0,Predicted 1
Actual 0,59,23
Actual 1,7,95


In [5]:
get_classification_report(final_lr_model, X_test, y_test)

,precision,recall,f1-score,support
0,0.893939,0.719512,0.797297,82.000000
1,0.805085,0.931373,0.863636,102.000000
accuracy,0.836957,0.836957,0.836957,0.836957
macro avg,0.849512,0.825442,0.830467,184.000000
weighted avg,0.844683,0.836957,0.834072,184.000000


In [6]:
get_feature_coefficients(final_lr_model)

,feature,coefficient,odds_ratio
0,categorical__slope_flat,0.998766,2.714929
1,categorical__cp_asymptomatic,0.957831,2.606037
2,categorical__missingindicator_slope_False,0.884789,2.422474
3,categorical__missingindicator_ca_True,0.756739,2.131314
4,categorical__sex_Male,0.699362,2.012467
5,categorical__missingindicator_fbs_True,0.666307,1.947033
6,categorical__exang_True,0.581967,1.789555
7,numeric__oldpeak,0.569082,1.766644
8,categorical__thal_reversable defect,0.439140,1.551372
9,categorical__ca_2.0,0.318612,1.375218


In [7]:
errors = get_prediction_errors(final_lr_model, X_test, y_test)

In [8]:
errors["error_type"].value_counts()

error_type
correct           154
false_positive     23
false_negative      7
Name: count, dtype: int64

In [9]:
false_positive = errors[errors["error_type"] == "false_positive"]
false_negative = errors[errors["error_type"] == "false_negative"]

In [10]:
false_positive

,age,trestbps,chol,thalch,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,actual,predicted,probability,error_type
5,59,130.0,318.0,120.0,1.0,Male,non-anginal,False,normal,True,flat,NaN,normal,0,1,0.742391,false_positive
14,72,160.0,NaN,114.0,1.6,Male,non-anginal,None,lv hypertrophy,False,flat,2.0,None,0,1,0.960783,false_positive
20,60,180.0,NaN,140.0,1.5,Male,non-anginal,False,st-t abnormality,True,flat,NaN,None,0,1,0.948523,false_positive
32,55,120.0,270.0,140.0,NaN,Male,asymptomatic,False,normal,False,None,NaN,None,0,1,0.725735,false_positive
33,58,100.0,213.0,110.0,NaN,Male,asymptomatic,False,st-t abnormality,False,None,NaN,None,0,1,0.762238,false_positive
47,51,128.0,NaN,107.0,NaN,Male,asymptomatic,False,normal,False,None,NaN,None,0,1,0.847430,false_positive
49,64,110.0,211.0,144.0,1.8,Male,typical angina,False,lv hypertrophy,True,flat,0.0,normal,0,1,0.640200,false_positive
50,37,130.0,315.0,158.0,NaN,Male,asymptomatic,False,normal,False,None,NaN,None,0,1,0.641915,false_positive
58,54,150.0,365.0,134.0,1.0,Male,asymptomatic,False,st-t abnormality,False,upsloping,NaN,None,0,1,0.767728,false_positive
70,59,135.0,234.0,161.0,0.5,Male,asymptomatic,False,normal,False,flat,0.0,reversable defect,0,1,0.524397,false_positive


In [11]:
false_negative

,age,trestbps,chol,thalch,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,actual,predicted,probability,error_type
10,57,154.0,232.0,164.0,NaN,Male,atypical angina,False,lv hypertrophy,False,upsloping,1.0,normal,1,0,0.305032,false_negative
19,32,95.0,NaN,127.0,0.7,Male,typical angina,None,normal,False,upsloping,NaN,None,1,0,0.333941,false_negative
31,40,152.0,223.0,181.0,NaN,Male,asymptomatic,False,normal,False,upsloping,0.0,reversable defect,1,0,0.343150,false_negative
39,56,120.0,279.0,150.0,1.0,Female,atypical angina,False,normal,False,flat,NaN,None,1,0,0.186608,false_negative
105,34,115.0,NaN,154.0,0.2,Male,asymptomatic,None,None,False,upsloping,NaN,None,1,0,0.460415,false_negative
128,62,NaN,NaN,NaN,NaN,Male,non-anginal,True,st-t abnormality,None,None,NaN,None,1,0,0.436017,false_negative
159,61,NaN,284.0,NaN,NaN,Male,non-anginal,False,normal,None,None,NaN,None,1,0,0.221186,false_negative


In [12]:
error_summary = errors.groupby("error_type")[NUMERIC_FEATURES].agg(
    ["count", "mean", "median"]
)

error_summary

age                   trestbps                     chol  \
               count       mean median    count        mean median count   
error_type                                                                 
correct          154  52.785714   53.0      147  132.442177  130.0   118   
false_negative     7  48.857143   56.0        5  127.200000  120.0     4   
false_positive    23  57.217391   58.0       22  134.000000  130.0    19   

                                  thalch                    oldpeak            \
                      mean median  count        mean median   count      mean   
error_type                                                                      
correct         246.474576  248.0    149  136.825503  135.0      76  1.751316   
false_negative  254.500000  255.5      5  155.200000  154.0       3  0.633333   
false_positive  254.789474  231.0     22  135.363636  140.0      12  1.391667   

                       
               median  
error_type             
correct          1.60  
false_negative   0.70  
false_positive   1.45

In [13]:
for column in CATEGORICAL_FEATURES:
    print(f"\n{column}")
    print(errors.groupby("error_type")[column].value_counts(normalize=True).round(3))


sex
error_type      sex   
correct         Male      0.786
                Female    0.214
false_negative  Male      0.857
                Female    0.143
false_positive  Male      0.957
                Female    0.043
Name: proportion, dtype: float64

cp
error_type      cp             
correct         asymptomatic       0.584
                atypical angina    0.208
                non-anginal        0.175
                typical angina     0.032
false_negative  asymptomatic       0.286
                atypical angina    0.286
                non-anginal        0.286
                typical angina     0.143
false_positive  asymptomatic       0.478
                non-anginal        0.348
                typical angina     0.130
                atypical angina    0.043
Name: proportion, dtype: float64

fbs
error_type      fbs  
correct         False    0.815
                True     0.185
false_negative  False    0.800
                True     0.200
false_positive  False    0.818
    

> Based on the results above we can draw some useful observation such as: there are 23 false positives and 7 false negatives, while false negatives in the test set generally have relatively low `oldpeak` and relatively high `thalch`, altough the sample is only 7 observations. The categorical distributions show some differences, but several group are too small to support strong conclusions. The error analysis is appropriately descriptive rather than claiming causality.

In [18]:
incorrect_predictions = errors[errors["error_type"] != "correct"].sort_values(
    "probability"
)

In [22]:
incorrect_predictions

,age,trestbps,chol,thalch,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,actual,predicted,probability,error_type
39,56,120.0,279.0,150.0,1.0,Female,atypical angina,False,normal,False,flat,NaN,None,1,0,0.186608,false_negative
159,61,NaN,284.0,NaN,NaN,Male,non-anginal,False,normal,None,None,NaN,None,1,0,0.221186,false_negative
10,57,154.0,232.0,164.0,NaN,Male,atypical angina,False,lv hypertrophy,False,upsloping,1.0,normal,1,0,0.305032,false_negative
19,32,95.0,NaN,127.0,0.7,Male,typical angina,None,normal,False,upsloping,NaN,None,1,0,0.333941,false_negative
31,40,152.0,223.0,181.0,NaN,Male,asymptomatic,False,normal,False,upsloping,0.0,reversable defect,1,0,0.343150,false_negative
128,62,NaN,NaN,NaN,NaN,Male,non-anginal,True,st-t abnormality,None,None,NaN,None,1,0,0.436017,false_negative
105,34,115.0,NaN,154.0,0.2,Male,asymptomatic,None,None,False,upsloping,NaN,None,1,0,0.460415,false_negative
70,59,135.0,234.0,161.0,0.5,Male,asymptomatic,False,normal,False,flat,0.0,reversable defect,0,1,0.524397,false_positive
147,59,180.0,213.0,100.0,NaN,Male,non-anginal,False,normal,False,None,NaN,None,0,1,0.574813,false_positive
157,62,120.0,220.0,86.0,NaN,Male,non-anginal,False,lv hypertrophy,False,None,NaN,None,0,1,0.598213,false_positive


In [19]:
incorrect_predictions[["actual", "predicted", "probability", "error_type"]]

,actual,predicted,probability,error_type
39,1,0,0.186608,false_negative
159,1,0,0.221186,false_negative
10,1,0,0.305032,false_negative
19,1,0,0.333941,false_negative
31,1,0,0.343150,false_negative
128,1,0,0.436017,false_negative
105,1,0,0.460415,false_negative
70,0,1,0.524397,false_positive
147,0,1,0.574813,false_positive
157,0,1,0.598213,false_positive


In [21]:
errors.groupby("error_type")["probability"].agg(
    ["count", "mean", "median", "min", "max"]
)

,count,mean,median,min,max
error_type,,,,,
correct,154,0.614728,0.759639,0.013161,0.996387
false_negative,7,0.326621,0.333941,0.186608,0.460415
false_positive,23,0.732145,0.697575,0.524397,0.960783
